<div style="background-color: #2ccece; padding: 20px; border-radius: 10px;">

<h1 style="color: black;">

NLP Project


</div>


In [27]:
import pandas as pd
import re
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity


## Step 1: Exploration

In [5]:
# Path to the reviews dataset
REVIEWS_PATH = r"data\TripAdvisorTrainingDataProject1\reviews83325.csv"
reviews = pd.read_csv(REVIEWS_PATH)

# Dataset overview
print("Shape:", reviews.shape)
print("Columns:", reviews.columns.tolist())
display(reviews.head(5))



C:\Users\sandr\AppData\Local\Temp\ipykernel_10784\1399121962.py:3: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(REVIEWS_PATH)


Shape: (340385, 21)
Columns: ['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,758953508,188467,A Treasure in Paris,9262311F3378F8CC4709DD4D92380278,We come back to this huge square every time we...,5,5/7/2020 03:16:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,755705747,188467,A place to take a breath from exploring Paris.,836BAA8786B81033412B950CF5BDA70C,The most beautiful square in Paris has a stran...,5,31/5/2020 19:36:17,2019-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,750396525,188467,Beautiful and elegant,8FEC7B1C9034428EFF3BF8728B036829,"Lovely architecture, looks different again to ...",4,11/3/2020 06:16:38,2020-03,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Random sample of reviews
display(reviews.sample(5, random_state=42))

,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
105478,127684269,292257,HIP HIP MARAIS....,DFA5F34122ECCC4687BE89DA025BF4D0,we spent a super day wandering this beautiful ...,5,11/4/2012 16:55:03,2012-04,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
312069,560967760,11459530,Expérience captivante,3D45A9C4263692474F29AC1BB578507F,Une super expérience: notre guide néerlandaise...,5,17/2/2018 05:19:20,2018-02,fr,Desktop,...,[],False,False,561402920.0,fr,19/2/2018 04:49:17,Responsable relations clients,Cultival,"Cher visiteur, \nNous sommes ravis que vous ay...",Owner Response
50928,286951586,188679,Must see epic and mysterious,CF1F9E57DF1CE41C73BE079901C32145,This cathedral is a world wonder the architec...,5,8/7/2015 20:13:42,2014-08,en,Mobile,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
161184,658072350,718765,Superbe,D6EEC8C74AF03E79C423E9FF9C7798BF,Léon est un très bon restaurant. Chaque fois q...,5,12/3/2019 14:51:19,2019-03,fr,Desktop,...,"[{""name"":""Service"",""value"":""5""},{""name"":""Value...",False,False,658298558.0,fr,13/3/2019 16:31:42,Responsable relations clients,Léon d,C'est un plaisir de vous faire plaisir :) A bi...,Owner Response
231041,385772697,2176008,Wonderful crepes,EF97218BDDA9231A64E2095EF78CB173,Our last lunch before leaving for the airport....,5,25/6/2016 10:12:54,2016-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# mIssing values
print("\nMissing values:")
print(reviews.isna().sum().sort_values(ascending=False).head(20))



Missing values:
owner_responder         300988
owner_connection        300745
owner_title             300744
owner_response          300744
owner_date_review       300744
owner_langue            300744
owner_id                300744
date_visit               19238
idauteur                   659
titre                        3
machine_translated           0
machine_translatable         0
id                           0
subratings                   0
idplace                      0
published_platform           0
langue                       0
date_review                  0
note                         0
review                       0
dtype: int64


In [9]:
# Distribution of review languages
if "langue" in reviews.columns:
    display(reviews["langue"].value_counts().head(10))

langue
en    153071
fr     99078
it     22912
es     21768
pt     19521
ru      5463
de      5237
ja      4124
nl      2835
ko      1133
Name: count, dtype: int64

## Step 2: Text preprocessing

1) Filter the reviews: english only

In [10]:

LANG_COL = "langue"
TEXT_COL = "review"
PLACE_ID_COL = "idplace"

# Filtrer in egnlish
reviews_en = reviews.copy()
if LANG_COL in reviews_en.columns:
    reviews_en = reviews_en[reviews_en[LANG_COL].astype(str).str.lower().eq("en")]

# Keep the rows with non-missing text and place_id
reviews_en = reviews_en.dropna(subset=[TEXT_COL, PLACE_ID_COL])
reviews_en[TEXT_COL] = reviews_en[TEXT_COL].astype(str)

print("English reviews shape:", reviews_en.shape)
display(reviews_en.head(5))

English reviews shape: (153071, 21)


,id,idplace,titre,idauteur,review,note,date_review,date_visit,langue,published_platform,...,subratings,machine_translated,machine_translatable,owner_id,owner_langue,owner_date_review,owner_connection,owner_responder,owner_response,owner_title
0,771569620,188467,February,F645CC9429E8A40EB1F5A487780EC683,Personally I think it is the most beautiful sq...,5,23/9/2020 11:14:11,2020-02,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,769814072,188467,Nice green square,AFFB511F21DF819776CB2F8013034382,We walked through this lovely park but did not...,4,11/9/2020 07:52:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,758953508,188467,A Treasure in Paris,9262311F3378F8CC4709DD4D92380278,We come back to this huge square every time we...,5,5/7/2020 03:16:32,2019-10,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,755705747,188467,A place to take a breath from exploring Paris.,836BAA8786B81033412B950CF5BDA70C,The most beautiful square in Paris has a stran...,5,31/5/2020 19:36:17,2019-06,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,750396525,188467,Beautiful and elegant,8FEC7B1C9034428EFF3BF8728B036829,"Lovely architecture, looks different again to ...",4,11/3/2020 06:16:38,2020-03,en,Desktop,...,[],False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN


2. Preprocessing of the text in the review column


In [15]:
STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Standard Cleaning:
    - lowercase
    - delete URLs
    - delete ponctuation and special chars 
    - delete numbers
    - delete stopwords
    - lemmatization
    - return cleaned text
    """
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # keep only letters and spaces
    text = re.sub(r"[^a-z\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # simplified tokenization
    tokens = text.split()

    # stopwords + lemmatization
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens if t not in STOPWORDS and len(t) > 2]

    return " ".join(tokens)

# Apply cleaning
reviews_en["clean_text"] = reviews_en[TEXT_COL].apply(clean_text)

# verify results
display(reviews_en[[PLACE_ID_COL, TEXT_COL, "clean_text"]].head(5))

,idplace,review,clean_text
0,188467,Personally I think it is the most beautiful sq...,personally think beautiful square paris well m...
1,188467,We walked through this lovely park but did not...,walked lovely park stay long square literally ...
2,188467,We come back to this huge square every time we...,come back huge square every time paris spectac...
3,188467,The most beautiful square in Paris has a stran...,beautiful square paris strange name mountain d...
4,188467,"Lovely architecture, looks different again to ...",lovely architecture look different paris squar...


## Step 3: Places representation

1. Group the reviews by place

Concatenate reviews per place: for each place (idplace), we merge all its cleaned english reviews into a single text called place_docs. This converts the dataset from “one row = one review” to “one row = one place,” which matches the project goal (recommend similar places, not similar reviews).
Why we do it: IR models like BM25 and TF-IDF work on a set of documents. Here, each place must be a document so the system can rank and recommend places based only on their reviews.

In [16]:
# Create place documents by concatenating all cleaned reviews for each place
place_docs = (
    reviews_en.groupby(PLACE_ID_COL)["clean_text"]
    .apply(lambda x: " ".join(x.tolist()))
    .reset_index()
    .rename(columns={"clean_text": "place_document"})
)

print("Nb de lieux (docs):", place_docs.shape[0])
display(place_docs.head(5))

Nb de lieux (docs): 1835


,idplace,place_document
0,188467,personally think beautiful square paris well m...
1,188468,old college friend booked beautiful expensive ...
2,188470,winter lot going however easy see cobblestone ...
3,188471,call passe partout shop serious understatement...
4,188472,old historical place attended experience conce...


2. Representation

- Strategy 1: Word-based truncation (Concatenate + cap to X words)

After building place_docs for each place, we keep only the first X words (e.g., 1200–1500 words) and discard the rest.
The number of reviews per place is highly variable. If we keep all reviews, very popular places produce extremely long documents that can dominate the ranking simply because they contain more terms. Truncating to a fixed maximum length reduces this length/popularity bias while still preserving enough content to describe the place.

In [17]:
def truncate_words(text: str, max_words: int = 1200) -> str:
    words = text.split()
    return " ".join(words[:max_words])

def apply_strategy_truncate(place_docs: pd.DataFrame, max_words: int = 1200) -> pd.DataFrame:
    out = place_docs.copy()
    out["place_document"] = out["place_document"].fillna("").apply(lambda t: truncate_words(t, max_words))
    return out

- Strategy 2:Top-K TF-IDF keywords 

We compute TF-IDF over the place documents, then for each place we select the K highest-weighted terms (e.g. K=200) and rebuild a reduced place_docs using only these keywords.
Why we do it: Instead of keeping all words (including noisy or generic ones), this strategy retains only the most discriminative terms that best characterize each place. It also normalizes document size across places, reducing the impact of places with many reviews.

In [19]:

def apply_strategy_topk_tfidf_words(place_docs: pd.DataFrame,
                                    k_words: int = 200,
                                    max_features: int = 50000,
                                    min_df: int = 2) -> pd.DataFrame:
    texts = place_docs["place_document"].fillna("").tolist()

    vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
    X = vectorizer.fit_transform(texts)
    vocab = np.array(vectorizer.get_feature_names_out())

    reduced_docs = []
    for i in range(X.shape[0]):
        row = X.getrow(i)
        if row.nnz == 0:
            reduced_docs.append("")
            continue
        idx = row.indices
        data = row.data
        top = idx[np.argsort(data)[::-1]][:k_words]
        reduced_docs.append(" ".join(vocab[top].tolist()))

    out = place_docs.copy()
    out["place_document"] = reduced_docs
    return out

Build the 2 strategies dataset

In [29]:
MAX_WORDS = 1200
TOPK_WORDS = 200
place_docs_S1 = apply_strategy_truncate(place_docs, max_words=MAX_WORDS)
place_docs_S2 = apply_strategy_topk_tfidf_words(place_docs, k_words=TOPK_WORDS)

print("Docs S1 shape:", place_docs_S1.shape)
print("Docs S2 shape:", place_docs_S2.shape)

Docs S1 shape: (1835, 2)
Docs S2 shape: (1835, 2)


## Step 4: Applying models

### 1. Baseline BM25 model:

In [33]:
def bm25_fit(place_docs_df):
    """
    place_docs_df: columns [idplace, place_document]
    returns: bm25_model, doc_ids, tokenized_docs
    """
    doc_ids = place_docs_df["idplace"].tolist()
    tokenized_docs = [doc.split() for doc in place_docs_df["place_document"].fillna("").tolist()]
    bm25 = BM25Okapi(tokenized_docs)
    return bm25, doc_ids, tokenized_docs

def bm25_rank(bm25, doc_ids, query_text, top_n=10, exclude_id=None):
    """
    Returns: list of (idplace, score) sorted desc.
    exclude_id: idplace to remove from results (usually the query place itself)
    """
    q_tokens = str(query_text).split()
    scores = bm25.get_scores(q_tokens)
    order = np.argsort(scores)[::-1]

    results = []
    for i in order:
        if exclude_id is not None and doc_ids[i] == exclude_id:
            continue
        results.append((doc_ids[i], float(scores[i])))
        if len(results) == top_n:
            break
    return results

### 2. TF-IDF + cosine model on place documents

In [35]:
def tfidf_fit(place_docs_df, max_features=20000, ngram_range=(1,2), min_df=2):
    """
    returns: vectorizer, X, doc_ids
    """
    doc_ids = place_docs_df["idplace"].tolist()
    texts = place_docs_df["place_document"].fillna("").tolist()
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range, min_df=min_df)
    X = vectorizer.fit_transform(texts)
    return vectorizer, X, doc_ids

def tfidf_rank(vectorizer, X, doc_ids, query_text, top_n=10, exclude_id=None):
    """
    Returns: list of (idplace, score) sorted desc.
    exclude_id: idplace to remove from results (usually the query place itself)
    """
    q = vectorizer.transform([str(query_text)])
    sims = cosine_similarity(q, X).ravel()
    order = np.argsort(sims)[::-1]

    results = []
    for i in order:
        if exclude_id is not None and doc_ids[i] == exclude_id:
            continue
        results.append((doc_ids[i], float(sims[i])))
        if len(results) == top_n:
            break
    return results

In [31]:
# Fit the 4 combinations
# BM25 on S1 / S2
bm25_S1, bm25_ids_S1, _ = bm25_fit(place_docs_S1)
bm25_S2, bm25_ids_S2, _ = bm25_fit(place_docs_S2)

# TF-IDF on S1 / S2
tfidf_vec_S1, X_S1, tfidf_ids_S1 = tfidf_fit(place_docs_S1, max_features=20000, ngram_range=(1,2), min_df=2)
tfidf_vec_S2, X_S2, tfidf_ids_S2 = tfidf_fit(place_docs_S2, max_features=20000, ngram_range=(1,1), min_df=1)

print("Models fitted: BM25_S1, BM25_S2, TFIDF_S1, TFIDF_S2")

Models fitted: BM25_S1, BM25_S2, TFIDF_S1, TFIDF_S2


Sanity check/debug

In [37]:
# Sanity check on 1 example place

# pick one place as query
q_row = place_docs_S1.sample(1, random_state=42).iloc[0]
q_id = q_row["idplace"]
q_text_S1 = q_row["place_document"]

# For S2, get the query text from place_docs_S2 for the same id
q_text_S2 = place_docs_S2.loc[place_docs_S2["idplace"] == q_id, "place_document"].values[0]

print("\nQuery place id:", q_id)
print("Query text (S1) preview:", q_text_S1[:200], "...\n")
print("Query text (S2) preview:", q_text_S2[:200], "...\n")

print("BM25 + S1 top10:", bm25_rank(bm25_S1, bm25_ids_S1, q_text_S1, top_n=10, exclude_id=q_id))
print("BM25 + S2 top10:", bm25_rank(bm25_S2, bm25_ids_S2, q_text_S2, top_n=10, exclude_id=q_id))

print("TFIDF + S1 top10:", tfidf_rank(tfidf_vec_S1, X_S1, tfidf_ids_S1, q_text_S1, top_n=10, exclude_id=q_id))
print("TFIDF + S2 top10:", tfidf_rank(tfidf_vec_S2, X_S2, tfidf_ids_S2, q_text_S2, top_n=10, exclude_id=q_id))


Query place id: 10377984
Query text (S1) preview: crossing seine ile cite saint jacques tower area single span iron bridge connecting stone arch either side carved stone head dionysus greek god especially like bridge pedestrian pavement meter wide hi ...

Query text (S2) preview: bridge notre dame arch cite stone pont ile bank one cathedral connects ram seine island span head connecting metal paris right pillar name view crossing iron crossed famous closest oldest dela natural ...

BM25 + S1 top10: [(1847871, 1029.5917332440295), (10291290, 1011.9947812803271), (2042830, 1011.5366141854339), (10171785, 978.6539646469092), (8408066, 937.486331488467), (3140972, 936.1642553113107), (10372975, 928.8285947414306), (8755696, 914.9759368378035), (10460878, 891.2372287441399), (10163562, 746.9324626757414)]
BM25 + S2 top10: [(10291290, 168.40291361405207), (2042830, 165.38125618681417), (10372975, 156.53546528152737), (8408066, 152.073860533224), (10163562, 137.55104231139563), (1847871, 136

For the query place 10377984, the four pipelines (BM25/TF-IDF × S1/S2) produce highly overlapping top-10 recommendations. This consistency is a good sanity check: it suggests the preprocessing and place-level document construction are working correctly, and that different lexical similarity methods retrieve broadly similar candidates.

Strategy S1 (truncate to max words) keeps a richer context (full sentences, more terms), which leads to higher TF-IDF cosine similarities (≈0.55–0.72) and very large BM25 scores (on an arbitrary scale). Strategy S2 (top-K TF-IDF keywords) compresses each place into a short list of discriminative terms. As expected, cosine similarities are lower (≈0.18–0.27) because documents share fewer tokens, but the retrieved places remain largely similar to S1, indicating that the most informative keywords preserve the main topic of the query (“bridge”, “Seine”, “Île”, “Notre-Dame”, etc.). Overall, these outputs confirm that the ranking functions run correctly; final model comparison should be done with the official Level-1/Level-2 ranking error metrics.

## Step 5:Evaluate with Ranking Error Level 1 and Level 2

Let's print the columns names

In [ ]:
tripadvisor = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\Tripadvisor.csv")
print("Tripadvisor columns:")
print(tripadvisor.columns.tolist())
reviews = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\reviews83325.csv")
print("Reviews columns:")
print(reviews.columns.tolist())
attr_subcat = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\AttractionSubCategorie.csv")
print("AttractionSubCategorie columns:")
print(attr_subcat.columns.tolist())
attr_subtype = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\AttractionSubType.csv")
print("AttractionSubType columns:")
print(attr_subtype.columns.tolist())
cuisine = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\cuisine.csv")
print("Cuisine columns:")
print(cuisine.columns.tolist())
restaurant_type = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\restaurantType.csv")
print("RestaurantType columns:")
print(restaurant_type.columns.tolist())
dietary = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\dietary_restrictions.csv")
print("Dietary restrictions columns:")
print(dietary.columns.tolist())

Tripadvisor columns:
['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview', 'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice', 'hotelBookingID', 'restaurantSubcategory', 'restaurantType', 'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids', 'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration', 'ap_exclusion', 'ap_inclusions', 'ap_introduction', 'ap_primary_supplier_attraction_id', 'ap_primary

C:\Users\sandr\AppData\Local\Temp\ipykernel_10784\2351612829.py:6: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(r"data\TripAdvisorTrainingDataProject1\reviews83325.csv")


Reviews columns:
['id', 'idplace', 'titre', 'idauteur', 'review', 'note', 'date_review', 'date_visit', 'langue', 'published_platform', 'typeReview', 'subratings', 'machine_translated', 'machine_translatable', 'owner_id', 'owner_langue', 'owner_date_review', 'owner_connection', 'owner_responder', 'owner_response', 'owner_title']
AttractionSubCategorie columns:
['id', 'name']
AttractionSubType columns:
['id', 'name']
Cuisine columns:
['id', 'name']
RestaurantType columns:
['id', 'name']
Dietary restrictions columns:
['id', 'name']


### 1. We start by building the metadata table that we will use only for evaluation

In [41]:
# Keep only useful columns for evaluation
meta_cols = [
    "id",
    "typeR",
    "priceRange",                     # hotels
    "restaurantTypeCuisine",          # restaurants
    "restaurantDietaryRestrictions",
    "activiteSubCategorie",           # attractions
    "activiteSubType"
]

meta = (
    tripadvisor[meta_cols]
    .rename(columns={"id": "idplace"})
    .drop_duplicates(subset=["idplace"])
)

print(meta.head())
print("Nb places with metadata:", meta.shape[0])

   idplace typeR priceRange restaurantTypeCuisine  \
0   188467     A        NaN                   NaN   
1   188468     A        NaN                   NaN   
2   188470     A        NaN                   NaN   
3   188471     A        NaN                   NaN   
4   188472     A        NaN                   NaN   

  restaurantDietaryRestrictions activiteSubCategorie activiteSubType  
0                           NaN                   47             163  
1                           NaN                   47             163  
2                           NaN             26,47,51          34,144  
3                           NaN                   26             137  
4                           NaN                   47              10  
Nb places with metadata: 3761


### 2. Split places into queries/candidates 50/50


In [42]:
def split_places_50_50(meta_df, seed=42):
    ids = meta_df["idplace"].dropna().unique()
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)

    mid = len(ids) // 2
    query_ids = set(ids[:mid])
    candidate_ids = set(ids[mid:])
    return query_ids, candidate_ids

query_ids, candidate_ids = split_places_50_50(meta)
print(len(query_ids), len(candidate_ids))

1880 1881


### 3. Ranking Error – Level 1 (typeR)

In [43]:
def ranking_error_L1(query_id, ranked_ids, meta_df):
    meta_idx = meta_df.set_index("idplace")

    if query_id not in meta_idx.index:
        return None

    query_type = meta_idx.loc[query_id, "typeR"]

    for pos, cid in enumerate(ranked_ids):
        if cid in meta_idx.index and meta_idx.loc[cid, "typeR"] == query_type:
            return pos
    return None

### 4. Ranking Error – Level 2 (detailed metadata)

In [46]:
def parse_list_field(x):
    """
    Helper function to convert '1|4|7' or '1,4,7' to a set of strings
    """
    if pd.isna(x):
        return set()
    if isinstance(x, str):
        return set([t.strip() for t in re.split(r"[|,]", x) if t.strip()])
    return set()

Match level 2

In [47]:
def match_L2(query_row, cand_row):
    t = query_row["typeR"]

    if t == "H":  # Hotel
        return query_row["priceRange"] == cand_row["priceRange"]

    if t == "R":  # Restaurant
        q_cuisine = parse_list_field(query_row["restaurantTypeCuisine"])
        c_cuisine = parse_list_field(cand_row["restaurantTypeCuisine"])
        return len(q_cuisine & c_cuisine) > 0

    if t in ["A", "AP"]:  # Attraction
        q_cat = parse_list_field(query_row["activiteSubCategorie"])
        q_sub = parse_list_field(query_row["activiteSubType"])
        c_cat = parse_list_field(cand_row["activiteSubCategorie"])
        c_sub = parse_list_field(cand_row["activiteSubType"])
        return len(q_cat & c_cat) > 0 or len(q_sub & c_sub) > 0

    return False

Ranking error L2

In [48]:
def ranking_error_L2(query_id, ranked_ids, meta_df):
    meta_idx = meta_df.set_index("idplace")

    if query_id not in meta_idx.index:
        return None

    q_row = meta_idx.loc[query_id]

    for pos, cid in enumerate(ranked_ids):
        if cid not in meta_idx.index:
            continue
        if match_L2(q_row, meta_idx.loc[cid]):
            return pos
    return None

### 5. Evaluation loop

In [49]:
def evaluate_model(query_ids, rank_function, meta_df, top_n=None):
    l1_errors = []
    l2_errors = []

    for qid in query_ids:
        ranked_ids = rank_function(qid)

        if top_n is not None:
            ranked_ids = ranked_ids[:top_n]

        e1 = ranking_error_L1(qid, ranked_ids, meta_df)
        e2 = ranking_error_L2(qid, ranked_ids, meta_df)

        if e1 is not None:
            l1_errors.append(e1)
        if e2 is not None:
            l2_errors.append(e2)

    return {
        "L1_mean_error": float(np.mean(l1_errors)) if l1_errors else None,
        "L2_mean_error": float(np.mean(l2_errors)) if l2_errors else None,
        "Nb_L1_queries": len(l1_errors),
        "Nb_L2_queries": len(l2_errors),
    }

#### Intersect ids 

In [53]:
docs_S1_all = set(place_docs_S1["idplace"].unique())
docs_S2_all = set(place_docs_S2["idplace"].unique())
docs_all = docs_S1_all.intersection(docs_S2_all)  # places available in both strategies

query_ids_eval = query_ids.intersection(docs_all)
candidate_ids_eval = candidate_ids.intersection(docs_all)

print("Query ids:", len(query_ids_eval), "Candidate ids:", len(candidate_ids_eval))

# Build quick lookup tables for query texts
docs_S1_query = place_docs_S1[place_docs_S1["idplace"].isin(query_ids_eval)].set_index("idplace")
docs_S2_query = place_docs_S2[place_docs_S2["idplace"].isin(query_ids_eval)].set_index("idplace")

# Candidate corpora
place_docs_S1_cand = place_docs_S1[place_docs_S1["idplace"].isin(candidate_ids_eval)].copy()
place_docs_S2_cand = place_docs_S2[place_docs_S2["idplace"].isin(candidate_ids_eval)].copy()

Query ids: 879 Candidate ids: 956


Fit models on candidates

In [55]:
# BM25 candidate indexes
bm25_S1_cand, bm25_ids_S1_cand, _ = bm25_fit(place_docs_S1_cand)  
bm25_S2_cand, bm25_ids_S2_cand, _ = bm25_fit(place_docs_S2_cand)

# TF-IDF candidate matrices
tfidf_vec_S1_cand, X_S1_cand, tfidf_ids_S1_cand = tfidf_fit(place_docs_S1_cand, max_features=20000, ngram_range=(1,2), min_df=2)
tfidf_vec_S2_cand, X_S2_cand, tfidf_ids_S2_cand = tfidf_fit(place_docs_S2_cand, max_features=20000, ngram_range=(1,1), min_df=1)

Rank functions

In [56]:
TOP_N_RANK = 500  # big enough so L2 can find a match

def rank_bm25_S1(qid):
    q_text = docs_S1_query.loc[qid, "place_document"]
    ranked = bm25_rank(bm25_S1_cand, bm25_ids_S1_cand, q_text, top_n=TOP_N_RANK)
    return [cid for cid, _ in ranked]

def rank_bm25_S2(qid):
    q_text = docs_S2_query.loc[qid, "place_document"]
    ranked = bm25_rank(bm25_S2_cand, bm25_ids_S2_cand, q_text, top_n=TOP_N_RANK)
    return [cid for cid, _ in ranked]

def rank_tfidf_S1(qid):
    q_text = docs_S1_query.loc[qid, "place_document"]
    ranked = tfidf_rank(tfidf_vec_S1_cand, X_S1_cand, tfidf_ids_S1_cand, q_text, top_n=TOP_N_RANK)
    return [cid for cid, _ in ranked]

def rank_tfidf_S2(qid):
    q_text = docs_S2_query.loc[qid, "place_document"]
    ranked = tfidf_rank(tfidf_vec_S2_cand, X_S2_cand, tfidf_ids_S2_cand, q_text, top_n=TOP_N_RANK)
    return [cid for cid, _ in ranked]

Evaluate all

In [ ]:
results = []

results.append(("BM25", "S1_truncate", evaluate_model(query_ids_eval, rank_bm25_S1, meta)))
results.append(("BM25", "S2_topk",     evaluate_model(query_ids_eval, rank_bm25_S2, meta)))
results.append(("TFIDF", "S1_truncate", evaluate_model(query_ids_eval, rank_tfidf_S1, meta)))
results.append(("TFIDF", "S2_topk",     evaluate_model(query_ids_eval, rank_tfidf_S2, meta)))


rows = []
for model_name, strat_name, res in results:
    rows.append({
        "Model": model_name,
        "Strategy": strat_name,
        "L1_mean_error": res["L1_mean_error"],
        "L2_mean_error": res["L2_mean_error"],
        "Nb_L1_queries": res["Nb_L1_queries"],
        "Nb_L2_queries": res["Nb_L2_queries"],
    })

results_df = pd.DataFrame(rows).sort_values(["Model", "Strategy"])

In [58]:
results_df

,Model,Strategy,L1_mean_error,L2_mean_error,Nb_L1_queries,Nb_L2_queries
0,BM25,S1_truncate,0.680319,2.796791,879,374
1,BM25,S2_topk,0.555176,5.469333,879,375
2,TFIDF,S1_truncate,0.749716,5.053191,879,376
3,TFIDF,S2_topk,0.645051,7.162234,879,376


### Results Interpretation

We evaluate the system using two ranking error levels. Level 1 measures whether the system retrieves a place with the same general type (hotel, restaurant, or attraction), while Level 2 measures similarity based on more detailed metadata such as cuisine type, activity subcategory, or price range. In both cases, a lower error means better performance.

For Level 1, all models obtain relatively low errors, between 0.55 and 0.75. This shows that the system usually retrieves a place of the correct type within the first one or two results. The best performance is achieved by BM25 with the Top-K keyword strategy (L1 = 0.56). This suggests that keeping only the most important keywords helps the model distinguish between different types of places.

For Level 2, the errors are higher, which is expected because fine-grained metadata cannot always be inferred directly from review text. The best result is obtained by BM25 with truncated documents (L2 = 2.80). This indicates that keeping more textual context allows the model to better capture detailed information, such as specific cuisines or activity subtypes. In comparison, the Top-K keyword strategy loses some of this information, which negatively impacts Level 2 performance, especially for TF-IDF.

Overall, the results are coherent and show a clear trade-off between the two strategies. Compact representations improve coarse type retrieval, while richer textual representations are more effective for fine-grained matching. Across all experiments, BM25 appears more robust than TF-IDF, particularly when detailed similarity is required.